In [3]:
# ─────────────────────────────────────────
# langchain_openai : LangChain용 OpenAI 래퍼
# 3번에서 직접 쓴 openai.OpenAI() 와 다르게
# LangChain 생태계에 맞게 감싸진 버전
# ─────────────────────────────────────────
import os
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool   # @tool 데코레이터
from dotenv import load_dotenv
load_dotenv(override=True)


# ─────────────────────────────────────────
# @tool 데코레이터
# 3번에서 register_tool(name, func, description) 로
# 직접 등록했던 것을 데코레이터 하나로 해결
#
# 핵심 규칙 2가지 :
# 1. 타입 힌트 필수 (query:str, price:float)
#    → LangChain이 이걸 보고 JSON Schema 자동 생성
# 2. docstring 필수 ('''...''')
#    → 이게 곧 Tool의 description
#    → LLM이 이 설명을 보고 언제 쓸지 판단
# ─────────────────────────────────────────
@tool
def search_knowledge(query: str) -> str:
    '''지식베이스를 검색합니다. 모르는 정보가 있을 때 사용하세요'''
    kb = {
        'LangChain': '랭체인은 LLM을 활용한 AI 애플리케이션을 쉽게 구축할 수 있는 오픈소스 프레임워크입니다.',
        'ReAct': 'ReAct는 추론(Reasoning)과 행동(Acting)을 결합한 프롬프팅 기법입니다.'
    }
    for key, val in kb.items():
        if key.lower() in query.lower():
            return val
    return '관련 정보를 찾을 수 없습니다.'

@tool
def string_length(text: str) -> int:
    '''문자열 길이를 반환합니다. 텍스트의 글자 수를 알아야 할 때 사용하세요'''
    return len(text)

@tool
def calculate_discount(price: float, discount_rate: float) -> float:
    '''원래 가격(price)과 할인율(discount_rate, 예 20%면 0.2)을 받아 할인 가격을 계산합니다'''
    return price * (1 - discount_rate)

# ─────────────────────────────────────────
# tools 리스트로 묶기
# 3번에서 registry.register() 로 하나씩 등록했던 것을
# 그냥 리스트로 관리
# ─────────────────────────────────────────
tools = [search_knowledge, string_length, calculate_discount]


# ─────────────────────────────────────────
# LLM 초기화
# 3번에서 client = OpenAI() 로 만든 것과 같은 역할
# 단, LangChain 생태계에서 쓰는 래퍼 클래스
# temperature=0 : 항상 일관된 형식으로 출력
# ─────────────────────────────────────────
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)


# ─────────────────────────────────────────
# create_agent : LLM + tools 를 결합해서
# ReAct 루프를 자동으로 실행하는 에이전트 생성
#
# 3번에서 직접 만들었던 것들을 한 줄로 대체
# ─────────────────────────────────────────
# SimpleReActAgent(3번)     create_agent(5번)
# ──────────────────────────────────────────
# __init__               →  내부적으로 처리
# register_tool()        →  tools 리스트로 전달
# _build_system_prompt() →  자동 생성
# _parse_action()        →  자동 파싱
# _execute_tool()        →  자동 실행
# run()                  →  agent.invoke()
# ──────────────────────────────────────────
from langchain.agents import create_agent
agent = create_agent(llm, tools)


# ─────────────────────────────────────────
# agent.invoke() : run() 과 같은 역할
# messages 형태로 입력
# 결과는 messages 리스트로 반환
# [-1] : 마지막 메시지 = 최종 답변
# ─────────────────────────────────────────

# 단일 Tool 사용
response1 = agent.invoke({
    'messages': [('user', '15000원짜리 물건을 15% 할인받으면 얼마인가요')]
})
print(f"단일 Tool:\n{response1['messages'][-1].content}")


# 다중 Tool 사용
# LangChain이 알아서 string_length + search_knowledge 두 개를 순서대로 호출
response2 = agent.invoke({
    'messages': [('user', 'LangChain이라는 단어의 글자 수는 몇 개인가요? 그리고 LangChain이 무엇인지 검색해주세요')]
})
print(f"\n다중 Tool:\n{response2['messages'][-1].content}")


# ─────────────────────────────────────────
# system 메시지 추가 예시
# 에이전트에 역할을 부여할 수 있음
# ─────────────────────────────────────────
response3 = agent.invoke({
    'messages': [
        ('system', '너는 부동산 최신 뉴스 전문 수집 및 분석가야'),
        ('user', '금천구 가산동 최신 부동산 실거래가를 알려주세요')
    ]
})
print(f"\n시스템 메시지 포함:\n{response3['messages'][-1].content}")

단일 Tool:
15000원짜리 물건을 15% 할인받으면 12750원이 됩니다.

다중 Tool:
"LangChain"이라는 단어의 글자 수는 9자입니다. 

LangChain은 LLM(대형 언어 모델)을 활용한 AI 애플리케이션을 쉽게 구축할 수 있는 오픈소스 프레임워크입니다.

시스템 메시지 포함:
금천구 가산동의 최신 부동산 실거래가에 대한 정보를 찾을 수 없습니다. 해당 정보를 확인하기 위해서는 부동산 관련 웹사이트나 정부의 부동산 실거래가 공개 시스템을 참고하시기 바랍니다. 추가로 궁금한 사항이 있으시면 말씀해 주세요!
